In [ ]:
!pip install rasterio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22.3/22.3 MB 42.1 MB/s eta 0:00:00


In [ ]:
import geopandas as gpd
import rasterio
import numpy as np
import pandas as pd
from shapely.geometry import Point

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
x_path = "/content/drive/MyDrive/GEE_Exports/samundraTapu_x_train_enhanced.tif"
lake_shapefile = "/content/drive/MyDrive/GEE_Exports/Shapefiles/samundraTapu/samundratapu_lake.shp"

In [ ]:
lakes = gpd.read_file(lake_shapefile)
lakes["class"] = 1   # mark as lakes

# Read raster (AOI extent)
with rasterio.open(x_path) as src:
    bounds = src.bounds
    crs = src.crs  # CRS of raster
    transform = src.transform

In [ ]:
np.random.seed(42)  # reproducibility
n_lakes = len(lakes)   # balance: one non-lake polygon per lake polygon
nonlake_polys = []


while len(nonlake_polys) < n_lakes:
    # Random coordinate within raster bounds
    x = np.random.uniform(bounds.left, bounds.right)
    y = np.random.uniform(bounds.bottom, bounds.top)
    point = Point(x, y)

    # Exclude points inside mapped lake polygons
    if not lakes.contains(point).any():
        # Create a small polygon buffer (30 m radius ~ 3 pixels in Sentinel-2)
        nonlake_polys.append(point.buffer(30))

In [ ]:
non_lakes = gpd.GeoDataFrame({"class": [0]*len(nonlake_polys)},
                              geometry=nonlake_polys, crs=crs)

In [ ]:
training = gpd.GeoDataFrame(pd.concat([lakes, non_lakes], ignore_index=True))
training.to_file("/content/drive/MyDrive/GEE_Exports/Shapefiles/samundraTapu/samundratapu_non_lake.shp")

print("✅ Training polygons saved")
print(training["class"].value_counts())

✅ Training polygons saved
class
1    1
0    1
Name: count, dtype: int64


In [ ]:
lake_shapefile

'/content/drive/MyDrive/GEE_Exports/Shapefiles/samundraTapu/samundratapu_lake.shp'